In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
import pickle
import os

print("🚀 Step 1: Generating Synthetic Labeled Dataset...")
# We generate 2000 synthetic Resume-JD pairs for training
np.random.seed(42)
n_samples = 2000

# Class 1: Good Matches (High skill overlap, high semantic similarity)
n_match = n_samples // 2
match_skills = np.random.normal(0.75, 0.15, n_match)
match_semantic = np.random.normal(0.80, 0.10, n_match)
match_lexical = np.random.normal(0.70, 0.15, n_match)
match_labels = np.ones(n_match)

# Class 0: Poor Matches (Low skill overlap, lower semantic similarity)
n_nomatch = n_samples // 2
nomatch_skills = np.random.normal(0.25, 0.20, n_nomatch)
nomatch_semantic = np.random.normal(0.40, 0.20, n_nomatch)
nomatch_lexical = np.random.normal(0.30, 0.20, n_nomatch)
nomatch_labels = np.zeros(n_nomatch)

# Combine and clip to [0, 1] range
X_match = np.column_stack((match_skills, match_semantic, match_lexical))
X_nomatch = np.column_stack((nomatch_skills, nomatch_semantic, nomatch_lexical))
X = np.vstack((X_match, X_nomatch))
X = np.clip(X, 0, 1) # Ensure all scores are strictly between 0 and 1
y = np.concatenate((match_labels, nomatch_labels))

# Create DataFrame
df = pd.DataFrame(X, columns=['skill_overlap_score', 'semantic_score', 'lexical_score'])
df['is_match'] = y

print("📊 Dataset Shape:", df.shape)

print("\n🚀 Step 2: Training the Supervised Meta-Learner (Random Forest)...")
X_train, X_test, y_train, y_test = train_test_split(df[['skill_overlap_score', 'semantic_score', 'lexical_score']], df['is_match'], test_size=0.2, random_state=42)

# Train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

# Predict on test set
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print("\n✅ Step 3: Research Evaluation Metrics")
print(f"Accuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred)*100:.2f}%")
print(f"Recall:    {recall_score(y_test, y_pred)*100:.2f}%")
print(f"F1-Score:  {f1_score(y_test, y_pred)*100:.2f}%")

print("\n🚀 Step 4: Generating Thesis Graphs...")
plt.style.use('default')
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Graph 1: Confusion Matrix
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Graph 2: ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc="lower right")

# Graph 3: Feature Importance
importances = rf_model.feature_importances_
features = ['Skill Overlap', 'Semantic Score', 'Lexical Score']
sns.barplot(x=features, y=importances, ax=axes[2], palette='viridis')
axes[2].set_title('Hybrid Feature Importance')
axes[2].set_ylabel('Weight in Final Decision')

plt.tight_layout()
plt.savefig('research_graphs_for_ppt.png', dpi=300)
plt.show()

print("\n🚀 Step 5: Exporting the Model for the FastAPI Backend...")
# We create a wrapper so the backend receives a probability (0-100%) instead of just a 0 or 1.
class ProbabilisticScorer:
    def __init__(self, model):
        self.model = model
    def predict(self, X):
        # Return the probability of class 1 (Match)
        return self.model.predict_proba(X)[:, 1]

final_scorer = ProbabilisticScorer(rf_model)

# Ensure ai_models directory exists
os.makedirs('ai_models', exist_ok=True)
with open('ai_models/mlp_scoring_model.pkl', 'wb') as f:
    pickle.dump(final_scorer, f)

print("✅ Model successfully saved to 'ai_models/mlp_scoring_model.pkl'!")
print("✅ The math formula in your backend has been overridden. The AI is now making the final decisions!")